# Baromètre ferroviaire SNCF — Dashboard interactif

**EuroMobilityDataHub · Bloc 2 — C2.1.3**

Dashboard d'analyse comparative de la ponctualité et des tarifs ferroviaires SNCF.  
Données : régularité mensuelle TGV / TER / Intercités + grilles tarifaires officielles SNCF (ODbL).

> **Prérequis** : exécuter le pipeline avant d'ouvrir ce notebook.
> ```bash
> ./.venv/bin/python pipeline/run.py --env preprod
> ```

**Usage :** `Kernel → Restart & Run All`, puis utiliser les widgets interactifs.

| Section | Contenu | Widget |
|---|---|---|
| 1 | Ponctualité par type de ligne | Slider de période |
| 2 | Distribution de la ponctualité | Sélecteur de types |
| 3 | Évolution temporelle — heatmap | — |
| 4 | Top & Bottom liaisons | Type + nombre |
| 5 | Prix au km vs ponctualité (H2) | — |
| 6 | Synthèse des tests statistiques (H1 + H2) | — |

In [ ]:
# ── Imports et configuration ────────────────────────────────────────────────────
import pathlib
import yaml
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from scipy import stats
import duckdb

ROOT = pathlib.Path('..').resolve()

COLORS = {
    'grande_vitesse': '#0072B2',
    'regional':       '#E69F00',
    'intercite':      '#009E73',
}
LABELS = {
    'grande_vitesse': 'Grande vitesse (TGV)',
    'regional':       'Régional (TER)',
    'intercite':      'Intercités',
}
ORDER = ['grande_vitesse', 'regional', 'intercite']
ENV = 'preprod'  # Changer ici pour 'dev' ou 'prod'


def _get_con() -> duckdb.DuckDBPyConnection:
    with open(ROOT / 'config' / f'{ENV}.yaml', encoding='utf-8') as _f:
        _cfg = yaml.safe_load(_f)
    db_path = ROOT / _cfg['database']['path']
    if not db_path.exists():
        raise FileNotFoundError(
            f'Base DuckDB introuvable : {db_path}\n'
            f'Exécuter : python pipeline/run.py --env {ENV}'
        )
    return duckdb.connect(str(db_path), read_only=True)


print(f'Environnement : {ENV}  |  Racine : {ROOT}')

In [ ]:
# ── Chargement des données ─────────────────────────────────────────────────────
_con = _get_con()

df_reg = _con.execute('''
    SELECT type_ligne, axe_label, mois,
           taux_ponctualite, taux_annulation,
           retard_moyen_tous_trains_arrivee_min,
           CAST(SUBSTR(mois, 1, 4) AS INTEGER) AS annee
    FROM fact_regularite
    WHERE taux_ponctualite IS NOT NULL
''').df()

df_fares = _con.execute('''
    SELECT type_ligne,
           axe_label_regularite AS axe_label,
           gare_origine, gare_destination,
           distance_km, prix_minimum, prix_maximum,
           prix_moyen, prix_moyen_km
    FROM fact_fares
    WHERE prix_moyen_km IS NOT NULL
      AND classe = '2'
      AND profil_tarifaire = 'Tarif Normal'
''').df()

_con.close()

ANNEES = sorted(df_reg['annee'].dropna().astype(int).unique().tolist())
_mois_min = df_reg['mois'].min()
_mois_max = df_reg['mois'].max()
print(f'Régularité : {len(df_reg):,} lignes  |  Tarifs : {len(df_fares):,} lignes')
print(f'Période : {_mois_min} → {_mois_max}  |  Années disponibles : {ANNEES[0]}–{ANNEES[-1]}')

## 1. Ponctualité par type de ligne
Comparaison de la ponctualité moyenne des trois types de ligne.  
**Utiliser le slider** pour restreindre la période d'analyse.

In [ ]:
_s1_slider = widgets.IntRangeSlider(
    value=[ANNEES[0], ANNEES[-1]],
    min=ANNEES[0], max=ANNEES[-1], step=1,
    description='Période :',
    continuous_update=False,
    layout=widgets.Layout(width='520px'),
    style={'description_width': 'initial'},
)
_out1 = widgets.Output()


def _draw_s1(change=None):
    a_min, a_max = _s1_slider.value
    df = df_reg[(df_reg['annee'] >= a_min) & (df_reg['annee'] <= a_max)]
    agg = df.groupby('type_ligne')['taux_ponctualite'].mean().round(1).reset_index()
    fig = go.Figure()
    for t in ORDER:
        row = agg[agg['type_ligne'] == t]
        if row.empty:
            continue
        v = float(row['taux_ponctualite'].iloc[0])
        fig.add_bar(
            x=[LABELS[t]], y=[v],
            name=LABELS[t],
            marker_color=COLORS[t],
            marker_line_color='black', marker_line_width=1.2,
            text=[f'{v:.1f} %'], textposition='outside',
        )
    fig.update_layout(
        title=f'Ponctualité moyenne par type de ligne ({a_min}–{a_max})',
        yaxis=dict(title='Taux de ponctualité moyen (%)', range=[0, 105]),
        showlegend=False, height=420, template='simple_white',
    )
    with _out1:
        _out1.clear_output(wait=True)
        fig.show()


_s1_slider.observe(_draw_s1, names='value')
display(_s1_slider, _out1)
_draw_s1()

## 2. Distribution de la ponctualité
Histogramme superposé — révèle la dispersion autour de la moyenne, pas seulement la moyenne.  
**Ctrl+clic** pour sélectionner / déselectionner plusieurs types.

In [ ]:
_s2_types = widgets.SelectMultiple(
    options=[(LABELS[t], t) for t in ORDER],
    value=list(ORDER),
    description='Types :',
    style={'description_width': 'initial'},
    layout=widgets.Layout(height='92px', width='280px'),
)
_out2 = widgets.Output()


def _draw_s2(change=None):
    selected = list(_s2_types.value)
    if not selected:
        return
    fig = go.Figure()
    for t in ORDER:
        if t not in selected:
            continue
        vals = df_reg[df_reg['type_ligne'] == t]['taux_ponctualite'].dropna().values
        fig.add_trace(go.Histogram(
            x=vals, name=LABELS[t],
            marker_color=COLORS[t], opacity=0.65,
            xbins=dict(start=0, end=100, size=4),
            marker_line_color='black', marker_line_width=0.7,
        ))
    fig.update_layout(
        barmode='overlay',
        title='Distribution du taux de ponctualité par type de ligne',
        xaxis_title='Taux de ponctualité (%)',
        yaxis_title='Nombre de (liaison/région × mois)',
        height=420, template='simple_white',
    )
    with _out2:
        _out2.clear_output(wait=True)
        fig.show()


_s2_types.observe(_draw_s2, names='value')
display(_s2_types, _out2)
_draw_s2()

## 3. Évolution temporelle — Heatmap
Tendance annuelle de la ponctualité par type de ligne.  
**Survoler** une cellule pour afficher la valeur exacte et l'année.

In [ ]:
_pivot = (
    df_reg
    .groupby(['type_ligne', 'annee'])['taux_ponctualite']
    .mean().round(1).reset_index()
    .pivot(index='type_ligne', columns='annee', values='taux_ponctualite')
)
_pivot = _pivot.loc[[t for t in ORDER if t in _pivot.index]]
_z = _pivot.values
_ylabels = [LABELS[t] for t in _pivot.index]
_xlabels = [str(int(a)) for a in _pivot.columns]
_text = [[f'{v:.0f}' if not pd.isna(v) else '' for v in row] for row in _z]

fig = go.Figure(data=go.Heatmap(
    z=_z, x=_xlabels, y=_ylabels,
    colorscale='Blues', zmin=50, zmax=100,
    text=_text, texttemplate='%{text}',
    textfont={'size': 11},
    colorbar=dict(title='Ponctualité (%)'),
    hoverongaps=False,
))
fig.update_layout(
    title='Évolution de la ponctualité par type de ligne et par année',
    xaxis_title='Année',
    height=290, template='simple_white',
)
fig.show()

## 4. Top & Bottom liaisons
Meilleures et moins bonnes liaisons par ponctualité moyenne (minimum 12 mois de données).  
*TER non inclus : agrégé par région, pas par liaison — non comparable au même grain (cf. limite 4.4 du dossier).*

In [ ]:
_s4_type = widgets.Dropdown(
    options=[
        ('Tous (hors TER)', 'all'),
        (LABELS['grande_vitesse'], 'grande_vitesse'),
        (LABELS['intercite'], 'intercite'),
    ],
    value='all',
    description='Type :',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='320px'),
)
_s4_n = widgets.IntSlider(
    value=5, min=3, max=15, step=1,
    description='Nb liaisons :',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'),
)
_out4 = widgets.Output()


def _draw_s4(change=None):
    t = _s4_type.value
    n = _s4_n.value
    df = df_reg[df_reg['type_ligne'] != 'regional'].copy()
    if t != 'all':
        df = df[df['type_ligne'] == t]
    agg = (
        df.groupby(['type_ligne', 'axe_label'])
        .agg(ponctualite=('taux_ponctualite', 'mean'),
             nb_mois=('taux_ponctualite', 'count'))
        .reset_index()
    )
    agg = agg[agg['nb_mois'] >= 12].copy()
    agg['ponctualite'] = agg['ponctualite'].round(1)
    top = agg.nlargest(n, 'ponctualite').assign(classement='Top')
    bottom = agg.nsmallest(n, 'ponctualite').assign(classement='Bottom')
    combined = pd.concat([top, bottom]).sort_values('ponctualite')
    fig = px.bar(
        combined, x='ponctualite', y='axe_label',
        color='type_ligne', color_discrete_map=COLORS,
        orientation='h', text='ponctualite',
        labels={'ponctualite': 'Ponctualité moyenne (%)', 'axe_label': 'Liaison / axe'},
        title=f'Top {n} et Bottom {n} liaisons par ponctualité (≥ 12 mois)',
    )
    fig.update_traces(texttemplate='%{text:.1f} %', textposition='outside')
    for tr in fig.data:
        tr.name = LABELS.get(tr.name, tr.name)
    fig.update_layout(
        height=max(380, n * 58),
        template='simple_white', legend_title='Type de ligne',
        xaxis=dict(range=[0, 105]),
    )
    with _out4:
        _out4.clear_output(wait=True)
        fig.show()


_s4_type.observe(_draw_s4, names='value')
_s4_n.observe(_draw_s4, names='value')
display(widgets.HBox([_s4_type, _s4_n]), _out4)
_draw_s4()

## 5. Prix au km vs Ponctualité — Test H2 (Spearman)
Chaque point est une liaison disposant à la fois d'un prix/km (grille tarifaire SNCF) et d'une ponctualité réelle.  
**Survoler** un point pour afficher le nom de la liaison, la distance et le prix exact.

In [ ]:
_ponct_agg = (
    df_reg
    .groupby(['type_ligne', 'axe_label'])['taux_ponctualite']
    .mean().reset_index()
    .rename(columns={'taux_ponctualite': 'ponctualite_moy'})
)
_df_h2 = (
    df_fares
    .merge(_ponct_agg, on=['type_ligne', 'axe_label'], how='inner')
    .query("type_ligne != 'regional'")
    .dropna(subset=['prix_moyen_km', 'ponctualite_moy'])
    .copy()
)
_df_h2['liaison'] = _df_h2['gare_origine'] + ' → ' + _df_h2['gare_destination']

if len(_df_h2) > 2:
    _rho, _p_h2 = stats.spearmanr(_df_h2['prix_moyen_km'], _df_h2['ponctualite_moy'])
    _titre_h2 = f'Prix au km vs Ponctualité  —  n={len(_df_h2)},  ρ={_rho:.3f},  p={_p_h2:.3g}'
else:
    _rho, _p_h2 = None, None
    _titre_h2 = f'Prix au km vs Ponctualité  —  n={len(_df_h2)} (échantillon insuffisant)'

fig = px.scatter(
    _df_h2,
    x='prix_moyen_km', y='ponctualite_moy',
    color='type_ligne', symbol='type_ligne',
    color_discrete_map=COLORS,
    hover_name='liaison',
    hover_data={
        'distance_km': ':.1f',
        'prix_moyen_km': ':.3f',
        'ponctualite_moy': ':.1f',
        'type_ligne': False,
    },
    labels={
        'prix_moyen_km': 'Prix moyen au km (€/km)',
        'ponctualite_moy': 'Ponctualité moyenne (%)',
    },
    title=_titre_h2,
)
fig.update_traces(marker=dict(size=10, line=dict(width=1, color='black')))
for tr in fig.data:
    tr.name = LABELS.get(tr.name, tr.name)
fig.update_layout(height=480, template='simple_white', legend_title='Type de ligne')
fig.show()

if _rho is not None:
    _verdict_h2 = 'H0 non rejetée — aucune corrélation significative' if _p_h2 >= 0.05 else 'H0 rejetée'
    print(f'\nH2 — Spearman : ρ = {_rho:.4f},  p = {_p_h2:.4g}  →  {_verdict_h2}')

## 6. Synthèse des tests statistiques
Résultats complets de H1 (ANOVA sur taux_ponctualite ~ type_ligne) et H2 (corrélation Spearman prix/km ~ ponctualité),  
avec statistiques descriptives par groupe.

In [ ]:
_groups_h1 = {
    t: df_reg[df_reg['type_ligne'] == t]['taux_ponctualite'].dropna().values
    for t in ORDER
}
_f_stat, _p_anova = stats.f_oneway(*_groups_h1.values())
_verdict_h1 = (
    'H0 rejetée : le type de ligne a un effet significatif sur la ponctualité'
    if _p_anova < 0.05 else 'H0 non rejetée'
)

print('=' * 65)
print('H1 — ANOVA : taux_ponctualite ~ type_ligne')
print(f'  F = {_f_stat:.2f}   p = {_p_anova:.2e}')
print(f'  → {_verdict_h1}')
print()
for t in ORDER:
    g = _groups_h1[t]
    print(f'  {LABELS[t]:30s}  n={len(g):5d}  moy={g.mean():.2f} %  σ={g.std():.2f}')

print()
print('=' * 65)
print('H2 — Spearman : prix_moyen_km ~ ponctualite_moy (par liaison)')
if _rho is not None:
    print(f'  n = {len(_df_h2)}   ρ = {_rho:.4f}   p = {_p_h2:.4g}')
    print(f'  → {_verdict_h2}')
else:
    print(f'  n = {len(_df_h2)} — échantillon insuffisant pour conclure.')
print('=' * 65)